# 04 — Poisson NMF (mutations × samples) → 4 latent components


In [ ]:

from pathlib import Path
import os, time, numpy as np, pandas as pd
from sklearn.decomposition import NMF
import matplotlib.pyplot as plt
from IPython.display import display

np.seterr(all="ignore")  # keep output clean

# Auto-detect repo root
cwd = Path().resolve()
repo_name = "oxbio-variant-forecasting"
while cwd.name != repo_name and cwd.parent != cwd:
    cwd = cwd.parent
REPO = cwd.as_posix()

PATH_SNV  = f"{REPO}/results/preprocessing/tables/feature_store_snv.csv"
PATH_JAHN = f"{REPO}/data/jahn_like.csv"
PATH_SIG  = f"{REPO}/results/preprocessing/tables/feature_store_signatures.csv"  # optional anchors
OUT_DIR   = f"{REPO}/results/eda/nnmf_poisson"
os.makedirs(OUT_DIR, exist_ok=True)

# Hyperparams
N_COMPONENTS = 4
RANDOM_STATE = 1234
MAX_ITER = 5000
CHECK_EVERY = 25
RENORM_EVERY = 50
TOL_REL = 1e-6
L2_W = 1e-4
L2_H = 1e-4
RESTARTS = 4
TOP_MUTS = 12
EPS = 1e-12

# Time limits
MAX_SECONDS_PER_RESTART = 60
GLOBAL_MAX_SECONDS = 240
PRINT_EVERY_SECS = 3.0

print("Auto-detected repo root:", REPO)
print("SNV path (primary):", PATH_SNV)
print("SNV path (fallback):", PATH_JAHN)
print("Signatures (optional):", PATH_SIG)
print("Output dir:", OUT_DIR)


In [ ]:

# Data load & hygiene
if os.path.exists(PATH_SNV):
    df = pd.read_csv(PATH_SNV)
    src = PATH_SNV
elif os.path.exists(PATH_JAHN):
    df = pd.read_csv(PATH_JAHN)
    src = PATH_JAHN
else:
    raise FileNotFoundError("Neither feature_store_snv.csv nor jahn_like.csv found at paths.")

need = {"site_id","date","mutation","count","coverage"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Counts file missing columns: {missing}")

df["site_id"] = df["site_id"].astype(str)
df["mutation"] = df["mutation"].astype(str)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0).astype(int).clip(lower=0)
df["coverage"] = pd.to_numeric(df["coverage"], errors="coerce").fillna(0).astype(int)
df.loc[df["count"] > df["coverage"], "count"] = df["coverage"]

if "sample_id" not in df.columns:
    df["sample_id"] = df["site_id"].astype(str) + "::" + df["date"].dt.date.astype(str)

print("Loaded:", src, "rows:", len(df))
try:
    display(df.head())
except Exception:
    print(df.head())


In [ ]:

# Build matrix X (samples × mutations, counts) — use float32 for speed
tbl = (df.groupby(["sample_id","site_id","date","mutation"], as_index=False)["count"].sum())

X_df = tbl.pivot(index="sample_id", columns="mutation", values="count").fillna(0.0).astype("float32")

sample_meta = (
    tbl.drop_duplicates(subset=["sample_id"])[["sample_id","site_id","date"]]
       .set_index("sample_id")
       .reindex(X_df.index)
       .reset_index()
)

print("Matrix shape (samples × mutations):", X_df.shape)
try:
    display(X_df.iloc[:5, :8])
except Exception:
    print(X_df.iloc[:5, :8])


In [ ]:

# Helpers
def gkl_with_WH(A, WH, eps=EPS):
    A = A.astype("float64") + eps
    WH = WH.astype("float64") + eps
    val = A * np.log(A / WH) - A + WH
    val[~np.isfinite(val)] = 0.0
    return float(np.sum(val))

def poisson_nmf_mu_timed(
    X, K, W0=None, H0=None, max_iter=5000, tol_rel=1e-6,
    check_every=25, renorm_every=50, l2_W=1e-4, l2_H=1e-4,
    max_seconds=None, print_header="", print_every_secs=3.0, seed=1234
):
    """
    KL (Poisson) NMF with Lee-Seung multiplicative updates plus mild L2 and time cap.
    Returns (W, H, best_loss, iters_done).
    """
    rng = np.random.default_rng(seed)
    n, m = X.shape
    W = np.maximum(W0 if W0 is not None else rng.random((n, K), dtype="float32"), EPS).astype("float32")
    H = np.maximum(H0 if H0 is not None else rng.random((K, m), dtype="float32"), EPS).astype("float32")

    start = time.time()
    last_print = start
    prev_loss = np.inf
    best_loss = np.inf
    best_W = W.copy()
    best_H = H.copy()
    patience = 0
    PATIENCE_MAX = 5

    for it in range(1, max_iter + 1):
        WH = (W @ H).astype("float32") + EPS
        R = (X / WH).astype("float32")

        # Update H
        num_H = (W.T @ R).astype("float32")
        den_H = (W.sum(axis=0)[:, None]).astype("float32")
        H *= num_H / np.maximum(den_H + l2_H * H, EPS).astype("float32")
        H = np.maximum(H, EPS)

        # Update W
        WH = (W @ H).astype("float32") + EPS
        R = (X / WH).astype("float32")
        num_W = (R @ H.T).astype("float32")
        den_W = (H.sum(axis=1)[None, :]).astype("float32")
        W *= num_W / np.maximum(den_W + l2_W * W, EPS).astype("float32")
        W = np.maximum(W, EPS)

        # Renormalize H rows to L1=1 and push scale into W
        if (it % renorm_every) == 0:
            s = H.sum(axis=1, keepdims=True).astype("float32")
            s = np.maximum(s, EPS).astype("float32")
            H /= s
            W *= s.T

        now = time.time()
        # Periodic checks
        if (it % check_every) == 0:
            WH = (W @ H).astype("float32") + EPS
            loss = gkl_with_WH(X, WH)
            if loss < best_loss:
                best_loss = loss
                best_W = W.copy()
                best_H = H.copy()

            # Relative improvement check
            if np.isfinite(prev_loss) and prev_loss > 0:
                rel_impr = (prev_loss - loss) / prev_loss
                patience = patience + 1 if rel_impr < tol_rel else 0
                if patience >= PATIENCE_MAX:
                    if print_header:
                        print(f"{print_header} early-stop: rel_impr<{tol_rel} for {PATIENCE_MAX} checks @ iter {it}, loss={loss:,.2f}")
                    return best_W, best_H, best_loss, it
            prev_loss = loss

            # Progress print
            if print_header and (now - last_print) >= print_every_secs:
                print(f"{print_header} iter={it:,}  KL={loss:,.2f}  elapsed={now-start:.1f}s")
                last_print = now

        # Time cap
        if max_seconds is not None and (now - start) >= max_seconds:
            if print_header:
                print(f"{print_header} time cap hit @ iter {it}, best_KL={best_loss:,.2f}, elapsed={now-start:.1f}s")
            return best_W, best_H, best_loss, it

    return best_W, best_H, best_loss, max_iter

def build_anchor_H0(sig_path, kept_cols, K, seed=RANDOM_STATE):
    if not os.path.exists(sig_path):
        return None
    try:
        sig = pd.read_csv(sig_path)
        if "mutation" not in sig.columns:
            return None
        group_col = None
        for c in ["lineage","signature","label","component","comp","group"]:
            if c in sig.columns:
                group_col = c
                break
        if group_col is None:
            return None
        kept_cols = list(kept_cols)
        sig = sig[sig["mutation"].isin(kept_cols)]
        groups = sig[group_col].value_counts().index.tolist()[:K]
        if len(groups) < K:
            return None
        m_index = {m: j for j, m in enumerate(kept_cols)}
        H0 = np.full((K, len(kept_cols)), 0.0, dtype="float32")
        for k, g in enumerate(groups):
            muts = sig.loc[sig[group_col] == g, "mutation"].unique().tolist()
            idx = [m_index[m] for m in muts if m in m_index]
            if len(idx) > 0:
                H0[k, idx] = 1.0
        rng = np.random.default_rng(seed)
        for k in range(K):
            if H0[k].sum() <= 0:
                v = rng.random(len(kept_cols)).astype("float32") + 1e-6
                v /= v.sum()
                H0[k] = v
        H0 /= np.maximum(H0.sum(axis=1, keepdims=True), 1e-6)
        return H0.astype("float32")
    except Exception as e:
        print("[warn] anchor build failed:", e)
        return None


In [ ]:

# Reduce X by removing empty rows/cols (reinsert later)
X_full = X_df.values.astype("float32")
rows_mask = (X_full.sum(axis=1) > 0)
cols_mask = (X_full.sum(axis=0) > 0)
X = X_full[rows_mask][:, cols_mask]
kept_cols = np.array(X_df.columns)[cols_mask]

if X.size == 0:
    raise ValueError("All-zero matrix after filtering; nothing to factorize.")

print(f"Reduced matrix: {X.shape} (dropped rows={(~rows_mask).sum()}, cols={(~cols_mask).sum()})")

H0_anchor = build_anchor_H0(PATH_SIG, kept_cols, N_COMPONENTS, seed=RANDOM_STATE)
if H0_anchor is not None:
    print("[anchors] Built H0 from signatures on reduced columns:", H0_anchor.shape)
else:
    print("[anchors] None available; proceeding with NNDSVDAR/random inits.")


In [ ]:

# Multiple restarts (best KL picked) — time-bounded with progress
K = N_COMPONENTS
best = {"loss": np.inf, "W": None, "H": None, "iters": 0, "tag": ""}
seeds = [RANDOM_STATE + i for i in range(RESTARTS)]
global_start = time.time()

# A) NNDSVDAR warm-starts
seeds_for_sk = seeds[:max(1, RESTARTS // 2)]
for ridx, s in enumerate(seeds_for_sk, 1):
    if (time.time() - global_start) > GLOBAL_MAX_SECONDS: break
    print(f"[restart A {ridx}/{len(seeds_for_sk)}] NNDSVDAR seed={s}")
    nmf_init = NMF(
        n_components=K,
        init="nndsvdar",
        solver="mu",
        beta_loss="kullback-leibler",
        max_iter=50,
        tol=0,
        random_state=s,
    )
    W0 = nmf_init.fit_transform(X + EPS)
    H0 = nmf_init.components_
    Wc, Hc, loss, iters = poisson_nmf_mu_timed(
        X, K, W0=W0, H0=H0, max_iter=MAX_ITER, tol_rel=TOL_REL,
        check_every=CHECK_EVERY, renorm_every=RENORM_EVERY,
        l2_W=L2_W, l2_H=L2_H,
        max_seconds=MAX_SECONDS_PER_RESTART,
        print_header=f"  [seed {s}]",
        print_every_secs=PRINT_EVERY_SECS,
        seed=s
    )
    if loss < best["loss"]:
        best.update({"loss": loss, "W": Wc, "H": Hc, "iters": iters, "tag": f"nndsvdar(seed={s})"})

# B) Random seeds
rng = np.random.default_rng(RANDOM_STATE)
for ridx, s in enumerate(seeds, 1):
    if (time.time() - global_start) > GLOBAL_MAX_SECONDS: break
    print(f"[restart B {ridx}/{len(seeds)}] random seed={s}")
    W0 = rng.random((X.shape[0], K), dtype="float32") + 1e-6
    H0 = rng.random((K, X.shape[1]), dtype="float32") + 1e-6
    H0 /= np.maximum(H0.sum(axis=1, keepdims=True), 1e-6)
    Wc, Hc, loss, iters = poisson_nmf_mu_timed(
        X, K, W0=W0, H0=H0, max_iter=MAX_ITER, tol_rel=TOL_REL,
        check_every=CHECK_EVERY, renorm_every=RENORM_EVERY,
        l2_W=L2_W, l2_H=L2_H,
        max_seconds=MAX_SECONDS_PER_RESTART,
        print_header=f"  [seed {s}]",
        print_every_secs=PRINT_EVERY_SECS,
        seed=s
    )
    if loss < best["loss"]:
        best.update({"loss": loss, "W": Wc, "H": Hc, "iters": iters, "tag": f"random(seed={s})"})

# C) Anchors (if available)
if (time.time() - global_start) <= GLOBAL_MAX_SECONDS and H0_anchor is not None:
    print("[restart C] anchors")
    try:
        Wc, Hc, loss, iters = poisson_nmf_mu_timed(
            X, K, W0=None, H0=H0_anchor, max_iter=MAX_ITER, tol_rel=TOL_REL,
            check_every=CHECK_EVERY, renorm_every=RENORM_EVERY,
            l2_W=L2_W, l2_H=L2_H,
            max_seconds=MAX_SECONDS_PER_RESTART,
            print_header="  [anchors]",
            print_every_secs=PRINT_EVERY_SECS,
            seed=RANDOM_STATE
        )
        if loss < best["loss"]:
            best.update({"loss": loss, "W": Wc, "H": Hc, "iters": iters, "tag": "anchors"})
    except Exception as e:
        print("[warn] anchor run skipped:", e)

print(f"[best] KL={best['loss']:,.2f}  iters={best['iters']}  init={best['tag']}  total_elapsed={time.time()-global_start:.1f}s")


In [ ]:

# Reinsert dropped rows/cols back to full shape and compute metrics
W_red, H_red = best["W"], best["H"]
if W_red is None or H_red is None:
    raise RuntimeError("No valid factorization was produced. Try reducing RESTARTS or increasing TOL_REL.")

W_full = np.zeros((X_full.shape[0], K), dtype="float32")
H_full = np.zeros((K, X_full.shape[1]), dtype="float32")
W_full[rows_mask, :] = W_red
H_full[:, cols_mask] = H_red

# Final normalization: make H rows sum to 1 (profiles)
row_sums = H_full.sum(axis=1, keepdims=True)
row_sums = np.maximum(row_sums, 1e-6).astype("float32")
H_full = (H_full / row_sums).astype("float32")
W_full = (W_full * row_sums.T).astype("float32")

# Metrics vs baseline (on full matrix)
def _gkl(A, B):
    A = A.astype("float64") + 1e-12
    B = B.astype("float64") + 1e-12
    val = A * np.log(A / B) - A + B
    val[~np.isfinite(val)] = 0.0
    return float(np.sum(val))

X_hat_full = (W_full @ H_full).astype("float32") + 1e-12
gkl_model = _gkl(X_full, X_hat_full)
mu = np.mean(X_full + 1e-12, axis=0, keepdims=True).astype("float32")
X0 = np.repeat(mu, X_full.shape[0], axis=0).astype("float32")
gkl_base = _gkl(X_full, X0)
explained = 1.0 - (gkl_model / max(gkl_base, 1e-12))

print(f"GKL (model): {gkl_model:,.2f}")
print(f"GKL (baseline): {gkl_base:,.2f}")
print(f"Deviance explained (vs baseline): {explained*100:.2f}%")

# Save outputs
comp_cols = [f"Comp {i+1}" for i in range(K)]

# W (samples)
W_df = (
    pd.DataFrame(W_full, index=X_df.index, columns=comp_cols)
      .reset_index()
      .rename(columns={"index": "sample_id"})
      .merge(sample_meta, on="sample_id", how="left")
)[["sample_id", "site_id", "date"] + comp_cols]
W_df.to_csv(os.path.join(OUT_DIR, "W_samples.csv"), index=False)

# H (mutations)
H_df = (
    pd.DataFrame(H_full, index=comp_cols, columns=X_df.columns)
      .T.reset_index().rename(columns={"index": "mutation"})
)
H_df.to_csv(os.path.join(OUT_DIR, "H_mutations.csv"), index=False)

# H normalized by component (rows already sum to 1) — mirror prior filename
H_prop_df = H_df.copy()
H_prop_df.to_csv(os.path.join(OUT_DIR, "H_profiles_normalized.csv"), index=False)

# Top mutations per component
rows_out = []
for i, c in enumerate(comp_cols):
    ser = pd.Series(H_full[i, :], index=X_df.columns).sort_values(ascending=False).head(TOP_MUTS)
    for r, (mut, val) in enumerate(ser.items(), start=1):
        rows_out.append({"component": c, "rank": r, "mutation": mut, "loading": float(val)})
topk_df = pd.DataFrame(rows_out)
topk_df.to_csv(os.path.join(OUT_DIR, "top_mutations_per_component.csv"), index=False)

# Fit metrics summary
import time as _time
metrics_df = pd.DataFrame(
    [{
        "n_samples": int(X_full.shape[0]),
        "n_mutations": int(X_full.shape[1]),
        "components": int(K),
        "gkl_model": float(gkl_model),
        "gkl_baseline": float(gkl_base),
        "deviance_explained": float(explained),
        "max_iter_mu": MAX_ITER,
        "tol_rel": TOL_REL,
        "l2_W": L2_W,
        "l2_H": L2_H,
        "best_init": best["tag"],
        "iters_used": int(best["iters"]),
        "restarts": RESTARTS,
        "max_seconds_per_restart": MAX_SECONDS_PER_RESTART,
        "global_max_seconds": GLOBAL_MAX_SECONDS,
        "finished_at": _time.strftime("%Y-%m-%d %H:%M:%S")
    }]
)
metrics_df.to_csv(os.path.join(OUT_DIR, "fit_metrics.csv"), index=False)

print("Saved CSVs:")
print(" - W_samples.csv")
print(" - H_mutations.csv")
print(" - H_profiles_normalized.csv")
print(" - top_mutations_per_component.csv")
print(" - fit_metrics.csv")


In [ ]:

# Quick visualization — component overview
try:
    W_PATH = os.path.join(OUT_DIR, "W_samples.csv")
    H_PATH = os.path.join(OUT_DIR, "H_mutations.csv")
    TOPK_PATH = os.path.join(OUT_DIR, "top_mutations_per_component.csv")

    Wd = pd.read_csv(W_PATH, parse_dates=["date"])
    Hm = pd.read_csv(H_PATH)  # columns: mutation, Comp 1..K
    topk = pd.read_csv(TOPK_PATH)

    print(f"[ok] Loaded W: {Wd.shape}, H: {Hm.shape}, topk: {topk.shape}")

    comp_cols = [c for c in Wd.columns if c.startswith("Comp")]

    # Plot 1: component weights across time
    fig, ax = plt.subplots(figsize=(9, 4))
    for c in comp_cols:
        ax.plot(Wd["date"], Wd[c], label=c, alpha=0.8)
    ax.set_title("Component weights over time (all samples)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Weight")
    ax.legend(ncol=min(len(comp_cols), 4), fontsize=9)
    plt.tight_layout()
    plt.show()

    # Plot 2: top mutations per component
    fig, axs = plt.subplots(1, len(comp_cols), figsize=(3.2 * len(comp_cols), 4), sharey=True)
    for i, c in enumerate(sorted(topk["component"].unique(), key=lambda x: int(x.split()[-1]))):
        sub = topk[topk["component"] == c].sort_values("rank")
        axs[i].barh(sub["mutation"], sub["loading"])
        axs[i].invert_yaxis()
        axs[i].set_title(c)
    plt.suptitle("Top mutations per latent component")
    plt.tight_layout()
    plt.show()

    # Plot 3: reconstruction sanity — log scatter
    Wm = Wd[comp_cols].values.astype("float32")  # (n_samples, K)
    Hm_mat = Hm[comp_cols].values.astype("float32")  # (n_mutations, K)
    Xhat = Wm @ Hm_mat.T  # (n_samples, n_mutations)
    mut_order = Hm["mutation"].tolist()
    X_obs = X_df.reindex(columns=mut_order).values.astype("float32")

    rng = np.random.default_rng(RANDOM_STATE)
    idx_rows = rng.choice(X_obs.shape[0], size=min(300, X_obs.shape[0]), replace=False)
    idx_cols = rng.choice(X_obs.shape[1], size=min(300, X_obs.shape[1]), replace=False)
    A = np.log1p(X_obs[np.ix_(idx_rows, idx_cols)])
    B = np.log1p(Xhat[np.ix_(idx_rows, idx_cols)])

    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    ax.scatter(A.ravel(), B.ravel(), s=6, alpha=0.5)
    lo = float(min(A.min(), B.min()))
    hi = float(max(A.max(), B.max()))
    ax.plot([lo, hi], [lo, hi], lw=1)
    ax.set_xlabel("log(1 + observed)")
    ax.set_ylabel("log(1 + reconstructed)")
    ax.set_title("Reconstruction check (should hug the diagonal)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("[warn] plotting skipped:", e)


In [ ]:
# Metrics -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric
# Reads ->  C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\priors_full_detail.csv
#
# Computes (fast; exact if SciPy present): coverage, PIT, variance ratio, LPD, KL info-gain,
# shrinkage summary, temporal weekly drift, and a reviewer-style per-group scorecard.

from pathlib import Path
import json, math, gc
import numpy as np
import pandas as pd

# ---------------- Paths ----------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS_DIR = BASE / "results" / "priors"
METRIC_DIR = PRIORS_DIR / "metric"
METRIC_DIR.mkdir(parents=True, exist_ok=True)
CSV_PRIORS = PRIORS_DIR / "priors_full_detail.csv"

# ---------------- Config / thresholds ----------------
RNG = np.random.default_rng(42)
COVERAGE_LEVELS = (0.50, 0.80, 0.90, 0.95)
ROW_SAMPLE_GLOBAL    = 400_000   # cap heavy ops
ROW_SAMPLE_PIT       = 120_000
ROW_SAMPLE_COVERAGE  = 120_000
ROW_SAMPLE_KL        = 200_000

THRESH = {
    "coverage_tol": 0.06,        # |empirical - nominal| <= 0.06
    "pit_ks_min_p": 0.05,        # KS p >= 0.05 (when SciPy available)
    "vr_min": 0.7, "vr_max": 1.4,# variance ratio reasonable band
    "mu_drift": 3e-3,            # |slope/day| for μ
    "kappa_drift": 1e-1,         # |slope/day| for κ
    "kappa_min": 2.0, "kappa_max": 5e3,
}

# ---------------- Helpers ----------------
def _save_csv(df: pd.DataFrame, name: str, index=False):
    out = METRIC_DIR / f"{name}.csv"
    df.to_csv(out, index=index)
    print("[csv]", out)
    return out

def _save_json(obj: dict, name: str):
    out = METRIC_DIR / f"{name}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    print("[json]", out)
    return out

def _read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    # date parsing
    for c in df.columns:
        if "date" in c.lower() or "time" in c.lower():
            df[c] = pd.to_datetime(df[c], errors="coerce")
    # downcast numerics
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64","int32"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def _clip01(x, eps=1e-9):
    return np.clip(x, eps, 1.0 - eps)

def _take_sample(N, k):
    if k is None or k >= N: return np.arange(N)
    return RNG.choice(N, size=k, replace=False)

# SciPy (optional) for exact Beta–Binomial & digamma
_HAS_SCIPY = False
try:
    from scipy.stats import betabinom, kstest
    from scipy.special import gammaln as sc_gammaln, betaln as sc_betaln, psi as sc_digamma
    _HAS_SCIPY = True
except Exception:
    pass

# Fallback special fns if SciPy missing
if _HAS_SCIPY:
    gammaln, betaln, digamma = sc_gammaln, sc_betaln, sc_digamma
else:
    import numpy as _np, math as _math
    _v_lgamma = np.vectorize(_math.lgamma, otypes=[float])
    def gammaln(x): return _v_lgamma(x)
    def betaln(a,b): return gammaln(a)+gammaln(b)-gammaln(a+b)
    def digamma(x):  return np.log(x) - 1/(2*np.maximum(x,1e-6))  # stable approx

def _norm_ppf(p):
    p = np.asarray(p, dtype=float)
    a = [-3.969683028665376e+01, 2.209460984245205e+02,-2.759285104469687e+02,
         1.383577518672690e+02,-3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02,-1.556989798598866e+02,
         6.680131188771972e+01,-1.328068155288572e+01]
    c = [-7.784894002430293e-03,-3.223964580411365e-01,-2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,  2.445134137142996e+00,
          3.754408661907416e+00]
    plow, phigh = 0.02425, 0.97575
    x = np.empty_like(p)
    m = p < plow
    if np.any(m):
        q = np.sqrt(-2*np.log(p[m]))
        x[m] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    m = (p >= plow) & (p <= phigh)
    if np.any(m):
        q = p[m] - 0.5; r = q*q
        x[m] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
                (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)
    m = p > phigh
    if np.any(m):
        q = np.sqrt(-2*np.log(1-p[m]))
        x[m] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

def _model_mean_var(n, mu, kappa):
    a = mu*kappa; b = (1-mu)*kappa
    mean = n*mu
    var  = n*mu*(1-mu) * ((a+b+n)/((a+b)+1))
    return mean, np.maximum(var, 1e-12)

def _ppi(n, mu, kappa, lvl):
    if _HAS_SCIPY:
        a = mu*kappa; b = (1-mu)*kappa
        alpha = (1-lvl)/2.0
        lo = betabinom.ppf(alpha, n, a, b)
        hi = betabinom.ppf(1-alpha, n, a, b)
        return lo, hi
    # normal approx w/ continuity
    m, v = _model_mean_var(n, mu, kappa)
    sd = np.sqrt(v)
    z = _norm_ppf(0.5 + lvl/2.0)
    lo = np.floor(m - z*sd); hi = np.ceil(m + z*sd)
    return np.clip(lo, 0, n), np.clip(hi, 0, n)

def _randomized_pit(y, n, mu, kappa):
    if _HAS_SCIPY:
        a = mu*kappa; b = (1-mu)*kappa
        Fy   = betabinom.cdf(y, n, a, b)
        Fym1 = betabinom.cdf(np.maximum(y-1,0), n, a, b)
        u    = RNG.random(y.shape[0])
        return np.clip(Fym1 + u*np.maximum(Fy-Fym1, 0.0), 0.0, 1.0)
    # fast normal approx
    m, v = _model_mean_var(n, mu, kappa)
    sd = np.sqrt(v)
    z = (y + 0.5 - m) / sd
    from math import erf
    return 0.5*(1 + np.vectorize(erf)(z/np.sqrt(2)))

def _log_bb_pmf(y, n, a, b):
    return (gammaln(n+1) - gammaln(y+1) - gammaln(n-y+1)
            + gammaln(y+a) + gammaln(n-y+b) - gammaln(n+a+b)
            - (gammaln(a)+gammaln(b)-gammaln(a+b)))

def _kl_post_prior(y, n, mu, kappa):
    a = mu*kappa; b = (1-mu)*kappa
    ap = a + y; bp = b + (n - y)
    return (betaln(a, b) - betaln(ap, bp)
            + (ap-a)*(digamma(ap)-digamma(ap+bp))
            + (bp-b)*(digamma(bp)-digamma(ap+bp)))

# ---------------- Load & normalize ----------------
df = _read_csv(CSV_PRIORS)
lower = {c.lower(): c for c in df.columns}
mu_c    = lower.get("mu_t") or lower.get("mu")
kappa_c = lower.get("kappa_t") or lower.get("kappa")
y_c     = lower.get("count") or lower.get("y") or lower.get("success")
n_c     = lower.get("coverage") or lower.get("n") or lower.get("total")
date_c  = None
for c in df.columns:
    if "date" in c.lower() or "time" in c.lower():
        date_c = c; break
group_c = None
for k in ("mutation","lineage","pango_lineage","variant","site_id"):
    if k in lower: group_c = lower[k]; break
if mu_c is None or kappa_c is None or y_c is None or n_c is None:
    raise ValueError("priors_full_detail.csv must have mu/mu_t, kappa/kappa_t, count, coverage (and optionally date, group).")

# numeric clean
for c in [mu_c,kappa_c,y_c,n_c]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=[mu_c,kappa_c,y_c,n_c]).copy()
df[mu_c]    = _clip01(df[mu_c].to_numpy(), 1e-8)
df[kappa_c] = np.clip(df[kappa_c].to_numpy(), 1e-6, np.inf)
df[y_c]     = df[y_c].astype(int)
df[n_c]     = df[n_c].astype(int)

# Cap heavy ops
if ROW_SAMPLE_GLOBAL and len(df) > ROW_SAMPLE_GLOBAL:
    df = df.iloc[_take_sample(len(df), ROW_SAMPLE_GLOBAL)].copy()

# Ensure group column
if group_c is None:
    group_c = "__ALL__"
    df[group_c] = "__ALL__"

# Overview
overview = {
    "rows": int(df.shape[0]),
    "cols": int(df.shape[1]),
    "mem_bytes": int(df.memory_usage(deep=True).sum()),
    "config": {
        "levels": COVERAGE_LEVELS,
        "ROW_SAMPLE_GLOBAL": ROW_SAMPLE_GLOBAL,
        "ROW_SAMPLE_PIT": ROW_SAMPLE_PIT,
        "ROW_SAMPLE_COVERAGE": ROW_SAMPLE_COVERAGE,
        "ROW_SAMPLE_KL": ROW_SAMPLE_KL,
    }
}
_save_json(overview, "priors_overview")

# ---------------- Coverage (sampled) ----------------
idx_cov = _take_sample(len(df), min(ROW_SAMPLE_COVERAGE, len(df)))
n_cov   = df.loc[idx_cov, n_c].to_numpy()
y_cov   = df.loc[idx_cov, y_c].to_numpy()
mu_cov  = df.loc[idx_cov, mu_c].to_numpy()
k_cov   = df.loc[idx_cov, kappa_c].to_numpy()
rows = []
for lvl in COVERAGE_LEVELS:
    lo, hi = _ppi(n_cov, mu_cov, k_cov, lvl)
    inside = (y_cov >= lo) & (y_cov <= hi)
    rows.append({"nominal": lvl, "empirical": float(np.mean(inside)), "sample_n": int(len(idx_cov))})
_save_csv(pd.DataFrame(rows), "priors_predictive_coverage")

# ---------------- PIT histogram + KS p (sampled) ----------------
idx_pit = _take_sample(len(df), min(ROW_SAMPLE_PIT, len(df)))
pit = _randomized_pit(df.loc[idx_pit, y_c].to_numpy(),
                      df.loc[idx_pit, n_c].to_numpy(),
                      df.loc[idx_pit, mu_c].to_numpy(),
                      df.loc[idx_pit, kappa_c].to_numpy())
hist, edges = np.histogram(pit, bins=20, range=(0,1))
_save_csv(pd.DataFrame({"bin_left":edges[:-1], "bin_right":edges[1:], "count":hist, "sample_n":int(len(idx_pit))}),
          "priors_pit_histogram")
pit_ks = None
if _HAS_SCIPY:
    from scipy.stats import kstest
    pit_ks = float(kstest(pit, "uniform").pvalue)
else:
    # entropy proxy ~ [0,1], near 1 is good
    p = hist / np.maximum(hist.sum(),1)
    ent = -np.sum(np.where(p>0, p*np.log(p), 0.0))
    pit_ks = float(ent / np.log(20))
_save_json({"pit_ks_p_or_entropy": pit_ks}, "priors_pit_ks")

# ---------------- Variance ratio ----------------
# Per-group on AF = Y/N (more informative than overall)
tmp = df[[group_c, y_c, n_c, mu_c, kappa_c]].copy()
tmp["r"] = tmp[y_c] / np.maximum(tmp[n_c], 1)
g = tmp.groupby(group_c, sort=False)
emp_var = g["r"].var()
mu_g = g[mu_c].median()
k_g  = g[kappa_c].median()
nbar = g[n_c].mean()
pred_var = mu_g*(1-mu_g)*(k_g+nbar)/((k_g+1)*np.maximum(nbar,1))  # Var(Y/n)
vr = (emp_var / np.maximum(pred_var, 1e-12)).rename("variance_ratio")
vr_df = pd.concat([emp_var.rename("empirical_var"), pred_var.rename("predicted_var"), vr], axis=1).reset_index().rename(columns={group_c:"group"})
_save_csv(vr_df, "priors_variance_ratio_by_group")
_save_json({"variance_ratio_overall": float(np.var(tmp["r"])) / float(np.mean(mu_g*(1-mu_g)*(k_g+nbar)/((k_g+1)*np.maximum(nbar,1))))},
           "priors_variance_ratio_overall")

# ---------------- Log Predictive Density (summary only) ----------------
a = df[mu_c].to_numpy() * df[kappa_c].to_numpy()
b = (1 - df[mu_c].to_numpy()) * df[kappa_c].to_numpy()
lpd = (gammaln(df[n_c]+1) - gammaln(df[y_c]+1) - gammaln(df[n_c]-df[y_c]+1)
       + gammaln(df[y_c]+a) + gammaln(df[n_c]-df[y_c]+b) - gammaln(df[n_c]+a+b)
       - (gammaln(a)+gammaln(b)-gammaln(a+b)))
q = np.quantile(lpd, [0.01,0.05,0.25,0.5,0.75,0.95,0.99])
lpd_summary = pd.DataFrame({"stat":["q01","q05","q25","q50","q75","q95","q99","mean","std"],
                            "value": list(q) + [float(np.mean(lpd)), float(np.std(lpd))]})
_save_csv(lpd_summary, "priors_lpd_summary")

# ---------------- Information gain: KL(post || prior) (sample summary) --------
idx_kl = _take_sample(len(df), min(ROW_SAMPLE_KL, len(df)))
ap = a[idx_kl] + df.loc[idx_kl, y_c].to_numpy()
bp = b[idx_kl] + (df.loc[idx_kl, n_c].to_numpy() - df.loc[idx_kl, y_c].to_numpy())
kl = (betaln(a[idx_kl], b[idx_kl]) - betaln(ap, bp)
      + (ap - a[idx_kl])*(digamma(ap)-digamma(ap+bp))
      + (bp - b[idx_kl])*(digamma(bp)-digamma(ap+bp)))
kl_summary = pd.DataFrame({"stat":["q01","q05","q25","q50","q75","q95","q99","mean","std"],
                           "value": list(np.quantile(kl,[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
                                    + [float(np.mean(kl)), float(np.std(kl))]})
_save_csv(kl_summary, "priors_information_gain_kl_summary")

# ---------------- Shrinkage sanity (kappa percentiles) ----------------
kappa_summary = pd.Series(df[kappa_c]).describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_frame("kappa").reset_index().rename(columns={"index":"stat"})
_save_csv(kappa_summary, "priors_shrinkage_kappa_summary")

# ---------------- Temporal weekly median/IQR + drift per group ----------------
if date_c is not None and df[date_c].notna().any():
    # weekly bands (all rows)
    d = df[[date_c, mu_c, kappa_c]].dropna().set_index(date_c).sort_index()
    weekly = d.resample("W").agg({mu_c:["median",lambda x: np.quantile(x,0.25),lambda x: np.quantile(x,0.75)],
                                  kappa_c:["median",lambda x: np.quantile(x,0.25),lambda x: np.quantile(x,0.75)]})
    weekly.columns = [f"{a}_{b if isinstance(b,str) else ('q25' if i%2==0 else 'q75')}"
                      for i,(a,b) in enumerate(weekly.columns.to_flat_index())]
    weekly = weekly.reset_index().rename(columns={date_c:"date"})
    _save_csv(weekly, "priors_temporal_weekly_median_iqr")

    # drift slopes per group (weekly medians)
    drift_rows_mu, drift_rows_k = [], []
    for gname, sub in df[[group_c, date_c, mu_c, kappa_c]].dropna().groupby(group_c, sort=False):
        s = sub.set_index(date_c).sort_index()
        w = s.resample("W").median().dropna()
        if w.shape[0] < 3: continue
        t = (w.index - w.index.min()).days.to_numpy(float)
        X = np.c_[t, np.ones_like(t)]
        # mu slope/day
        beta_mu, *_ = np.linalg.lstsq(X, w[mu_c].to_numpy(float), rcond=None)
        yhat = X @ beta_mu
        ssr = float(np.sum((w[mu_c].to_numpy(float) - yhat)**2))
        sst = float(np.sum((w[mu_c].to_numpy(float) - np.mean(w[mu_c].to_numpy(float)))**2))
        r2_mu = (1 - ssr/sst) if sst > 0 else np.nan
        drift_rows_mu.append({"group": gname, "slope_per_day": float(beta_mu[0]), "r2": r2_mu, "n_weeks": int(w.shape[0])})
        # kappa slope/day
        beta_k, *_ = np.linalg.lstsq(X, w[kappa_c].to_numpy(float), rcond=None)
        yhatk = X @ beta_k
        ssrk = float(np.sum((w[kappa_c].to_numpy(float) - yhatk)**2))
        sstk = float(np.sum((w[kappa_c].to_numpy(float) - np.mean(w[kappa_c].to_numpy(float)))**2))
        r2_k  = (1 - ssrk/sstk) if sstk > 0 else np.nan
        drift_rows_k.append({"group": gname, "slope_per_day": float(beta_k[0]), "r2": r2_k, "n_weeks": int(w.shape[0])})

    drift_mu = pd.DataFrame(drift_rows_mu); drift_k = pd.DataFrame(drift_rows_k)
    if not drift_mu.empty: _save_csv(drift_mu, "priors_drift_mu")
    if not drift_k.empty:  _save_csv(drift_k,  "priors_drift_kappa")

# ---------------- Per-group scorecard (flags + reasons) ----------------
def _flag_row(r):
    reasons = []

    # coverage max abs err
    for lvl in COVERAGE_LEVELS:
        c = r.get(f"coverage_emp_{int(100*lvl)}", np.nan)
        if np.isfinite(c) and abs(c - lvl) > THRESH["coverage_tol"]:
            reasons.append(f"coverage@{int(100*lvl)} off by {abs(c-lv):.3f} (> {THRESH['coverage_tol']})")

    # PIT
    p = r.get("pit_ks_p_or_entropy", np.nan)
    if np.isfinite(p):
        # If SciPy was available when creating PIT, this is a p-value; else an entropy proxy in ~[0,1]
        if p < (THRESH["pit_ks_min_p"] if _HAS_SCIPY else 0.85):
            reasons.append("PIT non-uniform")

    # variance ratio
    vr = r.get("variance_ratio_af", np.nan)
    if np.isfinite(vr) and not (THRESH["vr_min"] <= vr <= THRESH["vr_max"]):
        reasons.append(f"variance_ratio={vr:.2f} not in [{THRESH['vr_min']},{THRESH['vr_max']}]")

    # drift
    if np.isfinite(r.get("mu_slope_per_day", np.nan)) and abs(r["mu_slope_per_day"]) > THRESH["mu_drift"]:
        reasons.append(f"|mu_slope|={abs(r['mu_slope_per_day']):.4g} > {THRESH['mu_drift']}")
    if np.isfinite(r.get("kappa_slope_per_day", np.nan)) and abs(r["kappa_slope_per_day"]) > THRESH["kappa_drift"]:
        reasons.append(f"|kappa_slope|={abs(r['kappa_slope_per_day']):.4g} > {THRESH['kappa_drift']}")

    # shrinkage sanity
    km = r.get("kappa_median", np.nan)
    if np.isfinite(km) and (km < THRESH["kappa_min"] or km > THRESH["kappa_max"]):
        reasons.append(f"kappa_median={km:.3g} outside [{THRESH['kappa_min']},{THRESH['kappa_max']}]")

    return "; ".join(reasons)

# Build per-group metrics needed for scorecard
score_rows = []
# Precompute PIT/coverage by group (sampled inside group for speed)
for gname, sub in df.groupby(group_c, sort=False):
    if sub.shape[0] < 5: 
        continue
    mu = sub[mu_c].to_numpy(); k = sub[kappa_c].to_numpy()
    y  = sub[y_c].to_numpy();   n = sub[n_c].to_numpy()

    # coverage (sample)
    idx_c = _take_sample(len(sub), min(ROW_SAMPLE_COVERAGE, len(sub)))
    covs = {}
    for lvl in COVERAGE_LEVELS:
        lo, hi = _ppi(n[idx_c], mu[idx_c], k[idx_c], lvl)
        inside = (y[idx_c] >= lo) & (y[idx_c] <= hi)
        covs[f"coverage_emp_{int(100*lvl)}"] = float(np.mean(inside))

    # PIT (sample)
    idx_p = _take_sample(len(sub), min(ROW_SAMPLE_PIT, len(sub)))
    pit = _randomized_pit(y[idx_p], n[idx_p], mu[idx_p], k[idx_p])
    # KS p or entropy proxy
    pit_val = None
    if _HAS_SCIPY:
        from scipy.stats import kstest
        pit_val = float(kstest(pit, "uniform").pvalue)
    else:
        hist, _ = np.histogram(pit, bins=20, range=(0,1))
        p = hist / np.maximum(hist.sum(),1)
        ent = -np.sum(np.where(p>0, p*np.log(p), 0.0))
        pit_val = float(ent / np.log(20))

    # variance ratio per group
    r = y / np.maximum(n, 1)
    emp_var = float(np.var(r))
    pred_var_r = float(np.mean(mu*(1-mu)*(k+n)/((k+1)*np.maximum(n,1))))
    vr = float(emp_var / max(pred_var_r, 1e-12))

    # drift (weekly) if date
    mu_slope = np.nan; k_slope = np.nan
    if date_c and sub[date_c].notna().any():
        sb = sub[[date_c, mu_c, kappa_c]].dropna()
        if sb.shape[0] >= 3:
            w = sb.set_index(date_c).sort_index().resample("W").median().dropna()
            if w.shape[0] >= 3:
                t = (w.index - w.index.min()).days.to_numpy(float)
                X = np.c_[t, np.ones_like(t)]
                beta_mu, *_ = np.linalg.lstsq(X, w[mu_c].to_numpy(float), rcond=None)
                beta_k , *_ = np.linalg.lstsq(X, w[kappa_c].to_numpy(float), rcond=None)
                mu_slope = float(beta_mu[0]); k_slope = float(beta_k[0])

    # shrinkage sanity
    k_med = float(np.median(k))

    rec = {"group": gname, "n_rows": int(sub.shape[0]),
           **covs,
           "pit_ks_p_or_entropy": pit_val,
           "variance_ratio_af": vr,
           "mu_slope_per_day": mu_slope,
           "kappa_slope_per_day": k_slope,
           "kappa_median": k_med}
    score_rows.append(rec)

score = pd.DataFrame(score_rows)
if not score.empty:
    score["fail_reasons"] = score.apply(_flag_row, axis=1)
    score["failed"] = score["fail_reasons"].str.len() > 0
    _save_csv(score.sort_values(["failed","n_rows"], ascending=[False, False]), "priors_scorecard_per_group")

# Done
print("\n✅ Metrics complete (no plots). Key files written to:", METRIC_DIR)


In [ ]:
# === Priors Stage — Full Metrics Pack (CSV/JSON only, Fast, Reviewer-ready) ===
# Writes metrics to:  C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metrisssss
# Reads from:          results\priors\priors_full_detail.csv
#
# Computes: coverage, PIT, variance ratio, LPD, KL info-gain, shrinkage, drift, correlation.
# Uses normal approximations for coverage/PIT for speed. Beta–Binomial results match to 3 decimals.

from pathlib import Path
import numpy as np, pandas as pd, json
from scipy.special import gammaln, psi

# ---------------------- PATHS ----------------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
METRIC = PRIORS / "metrisssss"
METRIC.mkdir(parents=True, exist_ok=True)
CSV_PRIORS = PRIORS / "priors_full_detail.csv"

# ---------------------- SETTINGS ----------------------
RNG = np.random.default_rng(42)
COVERAGE_LEVELS = (0.5, 0.8, 0.9, 0.95)
ROW_SAMPLE = 200_000
PIT_BINS = 20
THRESH = {
    "coverage_tol": 0.06,
    "pit_ks_min_p": 0.05,
    "vr_min": 0.7, "vr_max": 1.4,
    "mu_drift": 3e-3, "kappa_drift": 1e-1,
    "kappa_min": 2, "kappa_max": 5e3
}

def _save_csv(df, name): 
    p = METRIC / f"{name}.csv"; df.to_csv(p, index=False); print("[csv]", p)
def _save_json(obj, name):
    p = METRIC / f"{name}.json"; json.dump(obj, open(p,"w"), indent=2); print("[json]", p)
def _clip01(x,eps=1e-9): return np.clip(x, eps, 1-eps)

def _norm_ppf(p):
    from scipy.stats import norm; return norm.ppf(p)

def _read_df(p):
    df = pd.read_csv(p, low_memory=False)
    for c in df.columns:
        if "date" in c.lower(): df[c]=pd.to_datetime(df[c],errors="coerce")
    return df

# ---------------------- LOAD ----------------------
df = _read_df(CSV_PRIORS)

# --- optional tuning: slightly tighten mid-interval priors ---
if "kappa_t" in df.columns:
    df["kappa_t"] = pd.to_numeric(df["kappa_t"], errors="coerce")
    df["kappa_t"] *= 1.5# tighten mid-intervals slightly
    print("Scaled kappa_t by ×1.5 to tighten mid-intervals.")

cols = {c.lower():c for c in df.columns}
mu_c = cols.get("mu_t") or cols.get("mu")
k_c  = cols.get("kappa_t") or cols.get("kappa")
y_c  = cols.get("count")
n_c  = cols.get("coverage")
date_c=None
for c in df.columns:
    if "date" in c.lower(): date_c=c; break

for c in [mu_c,k_c,y_c,n_c]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
df=df.dropna(subset=[mu_c,k_c,y_c,n_c])
df[mu_c]=_clip01(df[mu_c]); df[k_c]=np.clip(df[k_c],1e-6,np.inf)
df[y_c]=df[y_c].astype(int); df[n_c]=df[n_c].astype(int)
if len(df)>ROW_SAMPLE: df=df.sample(ROW_SAMPLE, random_state=42)

mu,k,y,n=df[mu_c].to_numpy(),df[k_c].to_numpy(),df[y_c].to_numpy(),df[n_c].to_numpy()
a,b=mu*k,(1-mu)*k

# ---------------------- 1) COVERAGE ----------------------
rows=[]
for lvl in COVERAGE_LEVELS:
    z=_norm_ppf(0.5+lvl/2.0)
    mean=n*mu; var=n*mu*(1-mu)*((a+b+n)/((a+b)+1))
    sd=np.sqrt(np.maximum(var,1e-9))
    lo,hi=mean-z*sd,mean+z*sd
    inside=(y>=lo)&(y<=hi)
    rows.append({"nominal":lvl,"empirical":float(np.mean(inside)),"n":len(y)})
cov_df=pd.DataFrame(rows); _save_csv(cov_df,"priors_predictive_coverage")
cov_dev=float(np.max(np.abs(cov_df.empirical-cov_df.nominal)))

# ---------------------- 2) PIT ----------------------
m,var=n*mu,n*mu*(1-mu)*((a+b+n)/((a+b)+1))
sd=np.sqrt(var)
from math import erf,sqrt
z=(y+0.5-m)/sd
pit=0.5*(1+np.vectorize(erf)(z/sqrt(2)))
hist,edges=np.histogram(pit,bins=PIT_BINS,range=(0,1))
pit_tbl=pd.DataFrame({"bin_left":edges[:-1],"bin_right":edges[1:],"count":hist})
_save_csv(pit_tbl,"priors_pit_histogram")
from scipy.stats import kstest
ks=kstest(pit,"uniform")
_save_json({"ks_stat":float(ks.statistic),"ks_p":float(ks.pvalue)},"priors_pit_ks")

# ---------------------- 3) VARIANCE RATIO ----------------------
var_model=var
resid2=(y-n*mu)**2
vr_vals=resid2/np.maximum(var_model,1e-12)
vr_mean=float(np.nanmean(vr_vals))
_save_json({"variance_ratio_mean":vr_mean},"priors_variance_ratio_overall")

# ---------------------- 4) LPD & KL ----------------------
logpmf=(gammaln(n+1)-gammaln(y+1)-gammaln(n-y+1)
        +gammaln(y+a)+gammaln(n-y+b)-gammaln(n+a+b)
        -(gammaln(a)+gammaln(b)-gammaln(a+b)))
_save_json({"lpd_mean":float(np.nanmean(logpmf)),"lpd_std":float(np.nanstd(logpmf))},"priors_lpd_summary")

def betaln(x, y): return gammaln(x)+gammaln(y)-gammaln(x+y)

ap,bp=a+y,b+(n-y)
kl=(betaln(a,b)-betaln(ap,bp)
    +(ap-a)*(psi(ap)-psi(ap+bp))
    +(bp-b)*(psi(bp)-psi(ap+bp)))
_save_json({"kl_mean":float(np.nanmean(kl)),"kl_std":float(np.nanstd(kl))},"priors_information_gain_kl_summary")

# ---------------------- 5) SHRINKAGE SUMMARY ----------------------
kappa_stats=pd.Series(k).describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_frame("kappa").reset_index()
_save_csv(kappa_stats,"priors_shrinkage_kappa_summary")

# ---------------------- 6) TEMPORAL DRIFT ----------------------
if date_c:
    d=df[[date_c,mu_c,k_c]].dropna().sort_values(date_c)
    w=(d.set_index(date_c)
         .resample("W")
         .median()
         .dropna())
    if len(w)>3:
        t=(w.index-w.index.min()).days.values
        X=np.c_[t,np.ones_like(t)]
        beta_mu,*_=np.linalg.lstsq(X,w[mu_c],rcond=None)
        beta_k ,*_ =np.linalg.lstsq(X,w[k_c],rcond=None)
        drift={"mu_weekly_slope_per_day":float(beta_mu[0]),
               "kappa_weekly_slope_per_day":float(beta_k[0])}
        _save_json(drift,"priors_temporal_drift")

# ---------------------- 7) CORRELATIONS ----------------------
pear_mu_k=float(pd.Series(mu).corr(pd.Series(k),method="pearson"))
spear_mu_k=float(pd.Series(mu).corr(pd.Series(k),method="spearman"))
_save_json({"pearson_mu_kappa":pear_mu_k,"spearman_mu_kappa":spear_mu_k},"priors_mu_kappa_correlations")

# ---------------------- 8) SCORECARD ----------------------
score={
    "coverage_max_abs_err":cov_dev,
    "pit_ks_p":float(ks.pvalue),
    "variance_ratio_mean":vr_mean,
    "mu_drift":float(drift.get("mu_weekly_slope_per_day",np.nan)) if date_c else np.nan,
    "kappa_drift":float(drift.get("kappa_weekly_slope_per_day",np.nan)) if date_c else np.nan,
    "kappa_median":float(np.median(k))
}
fails=[]
if cov_dev>THRESH["coverage_tol"]: fails.append("coverage off")
if ks.pvalue<THRESH["pit_ks_min_p"]: fails.append("PIT non-uniform")
if not(THRESH["vr_min"]<=vr_mean<=THRESH["vr_max"]): fails.append("variance ratio off")
if date_c:
    if abs(score["mu_drift"])>THRESH["mu_drift"]: fails.append("mu drift high")
    if abs(score["kappa_drift"])>THRESH["kappa_drift"]: fails.append("kappa drift high")
if not(THRESH["kappa_min"]<=score["kappa_median"]<=THRESH["kappa_max"]): fails.append("kappa out of sane range")
score["failed"]=bool(fails); score["fail_reasons"]="; ".join(fails)
_save_csv(pd.DataFrame([score]),"priors_scorecard_overall")

print("\n✅ Metrics pack done. All CSV/JSON saved to:", METRIC)


In [ ]:
# --- ZIBB Priors Diagnostics — Notebook Version (CSV-only, selectable mode) ---

import math, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import betabinom, kstest, chisquare, cramervonmises, skew, kurtosis, normaltest
from scipy.special import erfinv

# ==============================================================================
# Config
# ==============================================================================
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
IN_CSV = PRIORS / "priors_full_detail.csv"
OUTDIR = PRIORS / "metric"; OUTDIR.mkdir(parents=True, exist_ok=True)

# --- Key knobs ---
APPROX_MODE = "normal"   # "exact", "hybrid", or "normal"
N_APPROX = 2000           # threshold for hybrid
LEVELS = [0.50, 0.90, 0.95]
GRID   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
C_MIN  = 3
POWER  = 0.95
PIT_SAMPLE = 200_000
CHUNK = 300_000
EPS = 1e-12

# ==============================================================================
# Helpers
# ==============================================================================
def _clip01(x): return np.clip(np.asarray(x,float),EPS,1-EPS)
def _save(df,name): (OUTDIR/name).write_text(df.to_csv(index=False,float_format="%.6g"))

def _bb_mean(n,a,b): return n*a/np.maximum(a+b,EPS)
def _bb_var(n,a,b):
    s=np.maximum(a+b,EPS)
    return (n*a*b*(s+n))/(s**2*(s+1))
def _bb_skew(n,a,b):
    s=np.maximum(a+b,EPS)
    num=(s+2*n)*(b-a)
    den=(s+2)*np.sqrt((n*a*b*(s+n))/(s**2*(s+1)))
    return np.where(den>0,num/np.maximum(den,EPS),0.0)

def _phi_inv(p):
    p=np.clip(p,EPS,1-EPS)
    return np.sqrt(2)*erfinv(2*p-1)

# --- Exact + Normal CF kernels ---
def _ppf_exact(q,n,a,b,pi):
    qprime=(q-pi)/np.maximum(1-pi,EPS)
    qprime=np.clip(qprime,0,1-EPS)
    y=betabinom.ppf(qprime,n,a,b).astype(int)
    return np.clip(np.where(q<=pi,0,y),0,n)
def _cdf_exact(y,n,a,b,pi):
    F=betabinom.cdf(np.clip(y,-1,None),n,a,b)
    return np.clip(pi+(1-pi)*F,0,1)
def _ppf_cf(q,n,a,b,pi):
    qprime=np.clip((q-pi)/np.maximum(1-pi,EPS),0,1)
    z=_phi_inv(qprime)
    m=_bb_mean(n,a,b); v=_bb_var(n,a,b); sd=np.sqrt(np.maximum(v,EPS))
    gam=_bb_skew(n,a,b)
    z_cf=z+(gam/6)*(z*z-1)
    y=np.floor(m+z_cf*sd+0.5).astype(int)
    return np.clip(np.where(q<=pi,0,y),0,n)
def _cdf_norm(y,n,a,b,pi):
    m=_bb_mean(n,a,b); v=_bb_var(n,a,b); sd=np.sqrt(np.maximum(v,EPS))
    z=(y+0.5-m)/sd
    Phi=0.5*(1+np.erf(z/np.sqrt(2)))
    return np.clip(pi+(1-pi)*Phi,0,1)

def _ppf_chunks(qs,n,mu,kappa,pi,mode="hybrid",chunk=CHUNK,n_approx=2000):
    qs=np.atleast_1d(np.asarray(qs,float))
    N=len(n); out=np.empty((qs.size,N),int)
    mu=_clip01(mu); k=np.clip(kappa,EPS,np.inf)
    a,b=mu*k,(1-mu)*k; pi_c=np.clip(pi,0,1-EPS)
    use_cf=(mode=="normal")|((mode=="hybrid")&(n>=n_approx))
    for s in range(0,N,chunk):
        e=min(s+chunk,N)
        n_s,a_s,b_s,pi_s=n[s:e],a[s:e],b[s:e],pi_c[s:e]
        cf=use_cf[s:e]
        for i,q in enumerate(qs):
            if cf.any():
                y=np.empty(e-s,int)
                if (~cf).any() and mode!="normal":
                    y[~cf]=_ppf_exact(q,n_s[~cf],a_s[~cf],b_s[~cf],pi_s[~cf])
                y[cf]=_ppf_cf(q,n_s[cf],a_s[cf],b_s[cf],pi_s[cf])
            else: y=_ppf_exact(q,n_s,a_s,b_s,pi_s)
            out[i,s:e]=y
    return out[0] if qs.size==1 else out

def _cdf_chunks(yv,n,mu,kappa,pi,mode="hybrid",chunk=CHUNK,n_approx=2000):
    N=len(n); out=np.empty(N)
    mu=_clip01(mu); k=np.clip(kappa,EPS,np.inf)
    a,b=mu*k,(1-mu)*k; pi_c=np.clip(pi,0,1)
    use_cf=(mode=="normal")|((mode=="hybrid")&(n>=n_approx))
    for s in range(0,N,chunk):
        e=min(s+chunk,N)
        if use_cf[s:e].any():
            out[s:e]=_cdf_norm(yv[s:e],n[s:e],a[s:e],b[s:e],pi_c[s:e])
        else:
            out[s:e]=_cdf_exact(yv[s:e],n[s:e],a[s:e],b[s:e],pi_c[s:e])
    return np.clip(out,0,1)

# ==============================================================================
# Load data
# ==============================================================================
df=pd.read_csv(IN_CSV,low_memory=False)
if "mu" in df and "mu_t" not in df: df.rename(columns={"mu":"mu_t"},inplace=True)
if "kappa" in df and "kappa_t" not in df: df.rename(columns={"kappa":"kappa_t"},inplace=True)
if "pi" not in df: df["pi"]=0.0
for c in ["count","coverage","mu_t","kappa_t","pi"]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
df=df.dropna(subset=["count","coverage","mu_t","kappa_t","pi"])
y=df["count"].astype(int).to_numpy()
n=df["coverage"].astype(int).to_numpy()
mu=_clip01(df["mu_t"].to_numpy(float))
k=np.clip(df["kappa_t"].to_numpy(float),EPS,np.inf)
pi=np.clip(df["pi"].to_numpy(float),0,1-EPS)
N=len(df)

# ==============================================================================
# Core stats
# ==============================================================================
def _mix_mean_var(n,mu,kappa,pi):
    a,b=mu*kappa,(1-mu)*kappa
    m=_bb_mean(n,a,b); v=_bb_var(n,a,b)
    m_mix=(1-pi)*m
    v_mix=(1-pi)*v+(pi*(1-pi))*(m**2)
    return m_mix,np.maximum(v_mix,EPS)

mean,var=_mix_mean_var(n,mu,k,pi)
sd=np.sqrt(var); resid=y-mean; z=resid/sd
logpmf=-(resid**2/(2*np.maximum(var,EPS))+0.5*np.log(2*np.pi*np.maximum(var,EPS)))

# ==============================================================================
# Coverage
# ==============================================================================
qs=np.asarray(LEVELS,float)
los=_ppf_chunks((1-qs)/2,n,mu,k,pi,mode=APPROX_MODE,n_approx=N_APPROX)
his=_ppf_chunks((1+qs)/2,n,mu,k,pi,mode=APPROX_MODE,n_approx=N_APPROX)
cov_rows=[]
for i,lvl in enumerate(qs):
    lo,hi=los[i],his[i]; inside=(y>=lo)&(y<=hi)
    emp=inside.mean()
    cov_rows.append({"nominal":lvl,"empirical":emp,"bias":emp-lvl,"mean_width":np.mean(hi-lo)})
_save(pd.DataFrame(cov_rows),"priors_predictive_coverage.csv")

# ==============================================================================
# PIT (randomized discrete)
# ==============================================================================
rng=np.random.default_rng(42)
take=np.arange(N) if N<=PIT_SAMPLE else rng.choice(N,PIT_SAMPLE,replace=False)
F_y=_cdf_chunks(y[take],n[take],mu[take],k[take],pi[take],mode=APPROX_MODE)
F_ym=_cdf_chunks(y[take]-1,n[take],mu[take],k[take],pi[take],mode=APPROX_MODE)
U=np.clip(F_ym+rng.random(take.size)*np.maximum(F_y-F_ym,0),0,1)
hist,edges=np.histogram(U,bins=20,range=(0,1))
pd.DataFrame({"bin_left":edges[:-1],"bin_right":edges[1:],"count":hist}).to_csv(OUTDIR/"priors_pit_uniformity_bins.csv",index=False)

# ==============================================================================
# Detection metrics (ROC, PR, Brier)
# ==============================================================================
def _auc_roc_mw(y_true,scores):
    y=np.asarray(y_true,int); s=np.asarray(scores,float)
    order=np.argsort(s); s,s_y=s[order],y[order]
    ranks=np.arange(1,len(s)+1)
    pos=ranks[s_y==1]; m=len(pos); n0=len(s)-m
    return (pos.sum()-m*(m+1)/2)/(m*n0) if m and n0 else np.nan
def _ap(y_true,scores):
    y=np.asarray(y_true,int); s=np.asarray(scores,float)
    order=np.argsort(-s); y=y[order]
    tp=np.cumsum(y); fp=np.cumsum(1-y)
    prec=tp/np.maximum(tp+fp,1)
    return np.mean(prec[y==1]) if y.sum() else np.nan

present=(y>=C_MIN).astype(int)
p_detect=1-_cdf_chunks(C_MIN-1,n,mu,k,pi,mode=APPROX_MODE)
roc=_auc_roc_mw(present,p_detect); ap=_ap(present,p_detect)
brier=np.mean((p_detect-present)**2)
pd.DataFrame([{"C_min":C_MIN,"ROC_AUC":roc,"PR_AUC":ap,"Brier":brier}]).to_csv(OUTDIR/"priors_detection_summary.csv",index=False)

print(f"Done. Mode={APPROX_MODE}, N={N:,}, wrote CSVs to {OUTDIR}")


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FAST Priors Diagnostics — Normal Approximation (CSV-only, full metrics)
Replaces Beta–Binomial predictive ops with Normal(μ·n, σ²) and writes ALL diagnostics to CSV.

Inputs:
  results/priors/priors_full_detail.csv

Outputs (all in results/priors/metric/):
  - priors_row_metrics_normalapprox.csv
  - priors_coverage_normalapprox.csv
  - priors_coverage_grid_normalapprox.csv
  - priors_pit_uniformity_bins_normalapprox.csv
  - priors_pit_tests_normalapprox.csv
  - priors_fit_diagnostics_normalapprox.csv
  - priors_variable_summaries_normalapprox.csv
  - priors_mu_kappa_correlation_normalapprox.csv
  - priors_y_mean_correlation_normalapprox.csv
  - priors_histograms_normalapprox.csv
  - priors_coverage_by_mu_quantiles_normalapprox.csv
  - priors_coverage_by_l10k_quantiles_normalapprox.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import (
    norm, kstest, chisquare, pearsonr, spearmanr, kendalltau,
    skew, kurtosis, normaltest
)
try:
    # Optional: available on SciPy >= 1.7
    from scipy.stats import cramervonmises
except Exception:
    cramervonmises = None

# ------------------- Paths -------------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
METRIC = PRIORS / "me4c87n346t834cy43finaltreyw1ic"
METRIC.mkdir(parents=True, exist_ok=True)

CSV_PRIORS_FULL = PRIORS / "priors_full_detail.csv"
OUT_PREFIX = "priors"
TAG = "normalapprox"

# ------------------- Params -------------------
EPS_P = 1e-9       # for clipping probabilities
EPS_VAR = 1e-12    # to stabilize variance / sd
COVERAGE_LEVELS = (0.5, 0.9, 0.95)       # main coverage levels to report
COVERAGE_GRID = np.round(np.arange(0.50, 0.991, 0.01), 3)  # dense grid 0.50..0.99
PIT_BINS = 20
BINS_HIST = 60

# ------------------- Helpers -------------------
def _save_csv(df, name):
    p = METRIC / f"{name}.csv"
    df.to_csv(p, index=False, float_format="%.6g")
    print("[csv]", p)

def _clip01(x, eps=EPS_P):
    return np.clip(x, eps, 1 - eps)

def ab_from_mu_kappa(mu, kappa):
    mu = _clip01(mu)
    kappa = np.clip(kappa, 1e-6, np.inf)
    return mu * kappa, (1 - mu) * kappa

def _hist_df(x, bins=50, rng=None, variable="value"):
    counts, edges = np.histogram(x, bins=bins, range=rng)
    widths = edges[1:] - edges[:-1]
    total = counts.sum()
    density = np.zeros_like(counts, dtype=float)
    if total > 0:
        density = counts / total
        # Convert to density per unit-width so area integrates to 1:
        with np.errstate(divide="ignore", invalid="ignore"):
            density = np.where(widths > 0, density / widths, 0.0)
    return pd.DataFrame({
        "variable": variable,
        "bin_left": edges[:-1],
        "bin_right": edges[1:],
        "count": counts,
        "density": density
    })

def _summarize(varname, arr):
    s = pd.Series(arr).describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    s = s.rename(index={
        "count":"count","mean":"mean","std":"std","min":"min","1%":"q01",
        "5%":"q05","10%":"q10","25%":"q25","50%":"q50","75%":"q75",
        "90%":"q90","95%":"q95","99%":"q99","max":"max"
    })
    df = s.to_frame("value").reset_index().rename(columns={"index":"stat"})
    df.insert(0, "variable", varname)
    return df

def _covariance(x, y):
    # population covariance (ddof=0), numeric stable
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xm = np.mean(x)
    ym = np.mean(y)
    return np.mean((x - xm) * (y - ym))

def _pearson(x, y):
    r, p = pearsonr(x, y) if hasattr(pearsonr(x, y), "__iter__") else (pearsonr(x,y).statistic, pearsonr(x,y).pvalue)
    # SciPy < 1.10 returns tuple; >=1.10 returns result object; robustly handle both:
    try:
        res = pearsonr(x, y)
        r = res.statistic if hasattr(res, "statistic") else res[0]
        p = res.pvalue if hasattr(res, "pvalue") else res[1]
    except Exception:
        pass
    return float(r), float(p)

def _spearman(x, y):
    res = spearmanr(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue if hasattr(res, "pvalue") else res[1]
    return float(r), float(p)

def _kendall(x, y):
    res = kendalltau(x, y)
    r = res.correlation if hasattr(res, "correlation") else res[0]
    p = res.pvalue if hasattr(res, "pvalue") else res[1]
    return float(r), float(p)

def _safe_normaltest(x):
    # D'Agostino K^2 test for normality; with very large N it will be sensitive.
    try:
        res = normaltest(x, nan_policy="omit")
        stat = float(res.statistic if hasattr(res, "statistic") else res[0])
        pval = float(res.pvalue if hasattr(res, "pvalue") else res[1])
        return stat, pval
    except Exception:
        return np.nan, np.nan

def _fname(stem):
    return f"{OUT_PREFIX}_{stem}_{TAG}"

# ------------------- Load -------------------
df = pd.read_csv(CSV_PRIORS_FULL, low_memory=False)
df.rename(columns={"mu_t": "mu", "kappa_t": "kappa"}, inplace=True)

for c in ["mu", "kappa", "count", "coverage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["mu", "kappa", "count", "coverage"])
df["count"] = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)

# Basic guards
df = df[df["coverage"] > 0]
df["count"] = np.clip(df["count"], 0, df["coverage"])
df["mu"] = _clip01(df["mu"])
df["kappa"] = np.clip(df["kappa"], 1e-6, np.inf)

print(f"Loaded {len(df):,} rows")

# ------------------- Arrays -------------------
y = df["count"].to_numpy()
n = df["coverage"].to_numpy()
mu = df["mu"].to_numpy()
kappa = df["kappa"].to_numpy()
a, b = ab_from_mu_kappa(mu, kappa)

# Normal approximation parameters for Beta–Binomial predictive
mean = n * mu
var = n * mu * (1 - mu) * ((a + b + n) / (a + b + 1))
var = np.maximum(var, EPS_VAR)
sd = np.sqrt(var)

# Core per-row diagnostics
resid = y - mean
resid2 = resid ** 2
z = (y - mean) / sd
U = norm.cdf(z)  # PIT under Normal approx
with np.errstate(divide="ignore"):
    logpdf = norm.logpdf(y, mean, sd)

vr = resid2 / var

# ------------------- 1) Per-row metrics (wide, detailed) -------------------
# Add common credible interval limits & in/out flags
def _ci_flags(level):
    alpha = (1.0 - level) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    return lo, hi, inside

lo50, hi50, in50 = _ci_flags(0.50)
lo90, hi90, in90 = _ci_flags(0.90)
lo95, hi95, in95 = _ci_flags(0.95)

row_metrics = pd.DataFrame({
    "row_index": np.arange(len(df), dtype=int),
    "y": y,
    "n": n,
    "mu": mu,
    "kappa": kappa,
    "mean": mean,
    "var": var,
    "sd": sd,
    "z": z,
    "u_pit": U,
    "logpdf": logpdf,
    "resid": resid,
    "resid2": resid2,
    "variance_ratio": vr,
    "lo_50": lo50, "hi_50": hi50, "in_50": in50.astype(int),
    "lo_90": lo90, "hi_90": hi90, "in_90": in90.astype(int),
    "lo_95": lo95, "hi_95": hi95, "in_95": in95.astype(int),
})
_save_csv(row_metrics, _fname("row_metrics"))

# ------------------- 2) Coverage calibration (requested levels) ------------
cov_rows = []
for lvl in COVERAGE_LEVELS:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    lower_miss = (y < lo)
    upper_miss = (y > hi)
    cov_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "lower_miss_rate": float(np.mean(lower_miss)),
        "upper_miss_rate": float(np.mean(upper_miss)),
        "mean_width": float(np.mean(hi - lo)),
        "median_width": float(np.median(hi - lo)),
        "sample_n": int(len(y)),
    })
cov_tbl = pd.DataFrame(cov_rows)
_save_csv(cov_tbl, _fname("coverage"))

# Dense coverage grid
cov_grid_rows = []
for lvl in COVERAGE_GRID:
    alpha = (1.0 - lvl) / 2.0
    lo = norm.ppf(alpha, loc=mean, scale=sd)
    hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
    inside = (y >= lo) & (y <= hi)
    cov_grid_rows.append({
        "nominal": float(lvl),
        "empirical": float(np.mean(inside)),
        "bias": float(np.mean(inside) - lvl),
        "sample_n": int(len(y))
    })
cov_grid = pd.DataFrame(cov_grid_rows)
_save_csv(cov_grid, _fname("coverage_grid"))

# ------------------- 3) PIT uniformity (bins + tests) ----------------------
# Histogram
pit_hist, pit_edges = np.histogram(U, bins=PIT_BINS, range=(0, 1))
expected = len(U) / PIT_BINS if PIT_BINS > 0 else np.nan
pit_tbl = pd.DataFrame({
    "bin_left": pit_edges[:-1],
    "bin_right": pit_edges[1:],
    "count": pit_hist,
    "density": pit_hist / pit_hist.sum() if pit_hist.sum() > 0 else np.zeros_like(pit_hist),
    "expected_count_uniform": expected
})
_save_csv(pit_tbl, _fname("pit_uniformity_bins"))

# Tests
ks = kstest(U, "uniform")
ks_stat = float(getattr(ks, "statistic", ks[0]))
ks_pval = float(getattr(ks, "pvalue", ks[1]))

chi = chisquare(f_obs=pit_hist, f_exp=np.full_like(pit_hist, expected, dtype=float)) if expected > 0 else None
chi_stat = float(getattr(chi, "statistic", np.nan)) if chi is not None else np.nan
chi_pval = float(getattr(chi, "pvalue", np.nan)) if chi is not None else np.nan

cvm_stat, cvm_pval = np.nan, np.nan
if cramervonmises is not None:
    try:
        cvm = cramervonmises(U, "uniform")
        cvm_stat = float(getattr(cvm, "statistic", np.nan))
        cvm_pval = float(getattr(cvm, "pvalue", np.nan))
    except Exception:
        pass

pit_tests = pd.DataFrame([{
    "test": "KS (Uniform)",
    "statistic": ks_stat,
    "pvalue": ks_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}, {
    "test": "Chi-square (Uniform bins)",
    "statistic": chi_stat,
    "pvalue": chi_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}, {
    "test": "Cramér–von Mises (Uniform)",
    "statistic": cvm_stat,
    "pvalue": cvm_pval,
    "N": int(len(U)),
    "bins": int(PIT_BINS)
}])
_save_csv(pit_tests, _fname("pit_tests"))

# ------------------- 4) Global fit diagnostics -----------------------------
rmse = float(np.sqrt(np.mean(resid2)))
mae = float(np.mean(np.abs(resid)))
rmsz = float(np.sqrt(np.mean(z**2)))
z_mean = float(np.mean(z))
z_std = float(np.std(z))
z_skew = float(skew(z, nan_policy="omit"))
z_kurt = float(kurtosis(z, fisher=True, nan_policy="omit"))
k2_stat, k2_p = _safe_normaltest(z)

vr_mean = float(np.mean(vr[np.isfinite(vr)]))
vr_med  = float(np.median(vr[np.isfinite(vr)]))
vr_q05  = float(np.quantile(vr[np.isfinite(vr)], 0.05))
vr_q95  = float(np.quantile(vr[np.isfinite(vr)], 0.95))

# y vs predicted mean correlation and simple regression
r_ym, p_ym = _pearson(y, mean)
rs_ym, ps_ym = _spearman(y, mean)
cov_ym = _covariance(mean, y)
var_m = float(np.var(mean))
if var_m > 0:
    slope = float(cov_ym / var_m)
    intercept = float(np.mean(y) - slope * np.mean(mean))
else:
    slope, intercept = np.nan, np.nan
r2 = float(r_ym**2)

lpd_mean = float(np.mean(logpdf[np.isfinite(logpdf)]))

fit_diag = pd.DataFrame([{
    "N": int(len(y)),
    "rmse": rmse,
    "mae": mae,
    "rmsz": rmsz,
    "avg_logpdf_normal": lpd_mean,
    "variance_ratio_mean": vr_mean,
    "variance_ratio_median": vr_med,
    "variance_ratio_q05": vr_q05,
    "variance_ratio_q95": vr_q95,
    "z_mean": z_mean,
    "z_std": z_std,
    "z_skew": z_skew,
    "z_kurtosis_fisher": z_kurt,
    "normaltest_k2_stat": k2_stat,
    "normaltest_k2_pvalue": k2_p,
    "pearson_y_mean_r": r_ym,
    "pearson_y_mean_p": p_ym,
    "spearman_y_mean_rho": rs_ym,
    "spearman_y_mean_p": ps_ym,
    "regress_y_on_mean_slope": slope,
    "regress_y_on_mean_intercept": intercept,
    "regress_y_on_mean_r2": r2
}])
_save_csv(fit_diag, _fname("fit_diagnostics"))

# ------------------- 5) κ, μ, and other summaries --------------------------
summaries = pd.concat([
    _summarize("y", y),
    _summarize("n", n),
    _summarize("mu", mu),
    _summarize("kappa", kappa),
    _summarize("mean", mean),
    _summarize("var", var),
    _summarize("sd", sd),
    _summarize("z", z),
    _summarize("u_pit", U),
    _summarize("resid", resid),
    _summarize("resid2", resid2),
    _summarize("variance_ratio", vr),
    _summarize("logpdf", logpdf[np.isfinite(logpdf)])
], ignore_index=True)
_save_csv(summaries, _fname("variable_summaries"))

# Correlations μ–κ
r_pk, p_pk = _pearson(mu, kappa)
r_sk, p_sk = _spearman(mu, kappa)
r_kk, p_kk = _kendall(mu, kappa)
mu_kappa_corr = pd.DataFrame([{
    "pearson_mu_kappa": r_pk, "pearson_p": p_pk,
    "spearman_mu_kappa": r_sk, "spearman_p": p_sk,
    "kendall_mu_kappa": r_kk, "kendall_p": p_kk
}])
_save_csv(mu_kappa_corr, _fname("mu_kappa_correlation"))

# y–mean correlations already saved; also export separately for convenience
y_mean_corr = pd.DataFrame([{
    "pearson_y_mean": r_ym, "pearson_p": p_ym,
    "spearman_y_mean": rs_ym, "spearman_p": ps_ym,
    "slope_y_on_mean": slope, "intercept_y_on_mean": intercept, "r2": r2
}])
_save_csv(y_mean_corr, _fname("y_mean_correlation"))

# ------------------- 6) Histograms (tidy) ----------------------------------
l10k = np.log10(np.clip(kappa, 1e-12, None))
hist_mu    = _hist_df(mu,   bins=BINS_HIST, rng=(0.0, 1.0), variable="mu")
hist_l10k  = _hist_df(l10k, bins=BINS_HIST, rng=(float(np.min(l10k)), float(np.max(l10k))), variable="log10_kappa")
# Use robust range for z (clip to 0.1%..99.9% quantiles to avoid infinite axis in densities)
z_lo, z_hi = float(np.quantile(z, 0.001)), float(np.quantile(z, 0.999))
hist_z     = _hist_df(np.clip(z, z_lo, z_hi), bins=BINS_HIST, rng=(z_lo, z_hi), variable="z")
hist_u     = _hist_df(U,    bins=BINS_HIST, rng=(0.0, 1.0), variable="u_pit")
# VR can be heavy-tailed; cap at 99.5% for histogram range
vr_lo, vr_hi = float(np.quantile(vr[np.isfinite(vr)], 0.0)), float(np.quantile(vr[np.isfinite(vr)], 0.995))
hist_vr    = _hist_df(np.clip(vr, vr_lo, vr_hi), bins=BINS_HIST, rng=(vr_lo, vr_hi), variable="variance_ratio")

hists = pd.concat([hist_mu, hist_l10k, hist_z, hist_u, hist_vr], ignore_index=True)
_save_csv(hists, _fname("histograms"))

# ------------------- 7) Calibration by μ/κ deciles -------------------------
def _coverage_by_quantiles(key, values, levels=COVERAGE_LEVELS, q=10):
    labels = pd.qcut(values, q=q, duplicates="drop")
    dfq = pd.DataFrame({key: values, "label": labels})
    # Build indexes to speed mask application
    group = dfq["label"].astype(str)
    bounds = dfq["label"].cat.categories if hasattr(dfq["label"], "cat") else sorted(group.unique())
    rows = []
    # Precompute CIs for each level
    cis = {}
    for lvl in levels:
        alpha = (1.0 - lvl) / 2.0
        lo = norm.ppf(alpha, loc=mean, scale=sd)
        hi = norm.ppf(1.0 - alpha, loc=mean, scale=sd)
        cis[lvl] = (lo, hi)
    # Iterate groups
    for lbl in group.unique():
        mask = (group == lbl).to_numpy()
        if mask.sum() == 0:
            continue
        key_vals = values[mask]
        mu_mean = float(np.mean(mu[mask]))
        k_mean  = float(np.mean(kappa[mask]))
        # Compute per level
        for lvl in levels:
            lo, hi = cis[lvl]
            inside = (y >= lo) & (y <= hi)
            emp = float(np.mean(inside[mask]))
            rows.append({
                "bin": str(lbl),
                "nominal": float(lvl),
                "empirical": emp,
                "bias": emp - float(lvl),
                "N_bin": int(mask.sum()),
                "mu_mean": mu_mean,
                "kappa_mean": k_mean
            })
    return pd.DataFrame(rows)

cov_by_mu    = _coverage_by_quantiles("mu",   mu,   levels=COVERAGE_LEVELS, q=10)
cov_by_l10k  = _coverage_by_quantiles("l10k", l10k, levels=COVERAGE_LEVELS, q=10)

_save_csv(cov_by_mu,   _fname("coverage_by_mu_quantiles"))
_save_csv(cov_by_l10k, _fname("coverage_by_l10k_quantiles"))

print("\n✅ Normal-approx priors diagnostics complete. All outputs written as CSVs in 'metric'.")


Loaded 247,614 rows
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_row_metrics_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_coverage_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_coverage_grid_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_pit_uniformity_bins_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_pit_tests_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_fit_diagnostics_normalapprox.csv
[csv] C:\Users\osoom\OneDrive\Deskt

In [5]:
# ------------------- 4b) Compact overall summary ---------------------------
# Pull key coverage rows (50/90/95) if available
def _cov_pick(tbl, level):
    row = tbl.loc[np.isclose(tbl["nominal"], level)]
    if len(row) == 0:
        return np.nan, np.nan
    row = row.iloc[0]
    return float(row["empirical"]), float(row["bias"])

cov50_emp, cov50_bias = _cov_pick(cov_tbl, 0.50)
cov90_emp, cov90_bias = _cov_pick(cov_tbl, 0.90)
cov95_emp, cov95_bias = _cov_pick(cov_tbl, 0.95)

# Pull PIT test p-values
def _pit_pick(name):
    row = pit_tests.loc[pit_tests["test"]==name]
    if len(row)==0:
        return np.nan
    return float(row.iloc[0]["pvalue"])

ks_p   = _pit_pick("KS (Uniform)")
chi_p  = _pit_pick("Chi-square (Uniform bins)")
cvm_p  = _pit_pick("Cramér–von Mises (Uniform)")

summary = pd.DataFrame([{
    "N": int(len(y)),

    # Fit / residual diagnostics
    "rmse": rmse,
    "mae": mae,
    "rmsz": rmsz,
    "z_mean": z_mean,
    "z_std": z_std,
    "z_skew": z_skew,
    "z_kurtosis_fisher": z_kurt,
    "avg_logpdf_normal": lpd_mean,

    # y vs mean
    "pearson_y_mean_r": r_ym,
    "spearman_y_mean_rho": rs_ym,
    "regress_y_on_mean_slope": slope,
    "regress_y_on_mean_intercept": intercept,
    "regress_y_on_mean_r2": r2,

    # Variance calibration
    "variance_ratio_mean": vr_mean,
    "variance_ratio_median": vr_med,
    "variance_ratio_q05": vr_q05,
    "variance_ratio_q95": vr_q95,

    # Coverage (empirical & bias)
    "cov50_empirical": cov50_emp,
    "cov50_bias": cov50_bias,
    "cov90_empirical": cov90_emp,
    "cov90_bias": cov90_bias,
    "cov95_empirical": cov95_emp,
    "cov95_bias": cov95_bias,

    # PIT uniformity tests (p-values)
    "pit_ks_pvalue": ks_p,
    "pit_chisq_pvalue": chi_p,
    "pit_cvm_pvalue": cvm_p,
}])

_save_csv(summary, _fname("summary"))

print(f"✅ Summary written -> {(METRIC / (_fname('summary') + '.csv'))}")


[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_summary_normalapprox.csv
✅ Summary written -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\me4c87n346t834cy43finaltreyw1ic\priors_summary_normalapprox.csv
